# 07 — Preprocessing Workflow

This notebook demonstrates how **`quends`** ingests raw simulation output and prepares
it for statistical analysis.  The preprocessing pipeline covers:

1. Loading data from every supported source format
2. Inspecting available variables
3. Trimming away the transient start of a signal with the **strategy-operation** pattern
4. Assembling trimmed members into an `Ensemble`

> **Convention throughout these tutorials:** trimming is always performed via the
> explicit `QuantileTrimStrategy` / `TrimDataStreamOperation` API.  Never call
> `ds.trim()` directly.

## 1  Loading Data

`quends` ships four loader functions, all returning a `DataStream` object that
exposes a uniform API regardless of the original file format.

| Loader | Source |
|--------|--------|
| `qnds.from_csv(path)` | Comma-separated values file |
| `qnds.from_gx(path, variables=[...])` | GX NetCDF (`.nc`) simulation output |
| `qnds.from_dict(d)` | Python `dict` of array-like values |
| `qnds.from_numpy(arr, columns=[...])` | 2-D NumPy array |

### 1a  From CSV

In [ ]:
import quends as qnds

ds_csv = qnds.from_csv("gx/tprim_2_5_a.out.csv")
ds_csv.head()

### 1b  From GX NetCDF

GX produces NetCDF output (`.nc`).  Pass the path and an explicit list of
variables you need — this avoids loading large arrays you won't use.

In [ ]:
# Uncomment when a GX NetCDF file is available:
# ds_nc = qnds.from_gx("run.nc", variables=["time", "HeatFlux_st"])
# ds_nc.head()

### 1c  From a Python dict

Useful for synthetic data, unit tests, or wrapping arrays that are already
in memory.

In [ ]:
import numpy as np
import pandas as pd

data = {
    "time": np.linspace(0, 100, 500),
    "signal": np.random.randn(500).cumsum(),
}
ds_dict = qnds.from_dict(data)
ds_dict.head()

### 1d  From a NumPy array

Provide a 2-D array (rows = time steps, columns = variables) and supply
column names explicitly.

In [ ]:
arr = np.column_stack([
    np.linspace(0, 100, 500),
    np.random.randn(500),
])
ds_np = qnds.from_numpy(arr, columns=["time", "signal"])
ds_np.head()

---

## 2  Choosing Variables from a Large Dataset

GX output typically contains many diagnostics.  Use `.variables()` to see
what is available before committing to a column name — typos at this stage
will surface as `KeyError` later.

In [ ]:
# Inspect all available columns
print("Available variables:", ds_csv.variables())

# Work with a specific subset
col = "HeatFlux_st"
print(f"\nSelected column: '{col}'")
print(ds_csv.data[["time", col]].describe())

---

## 3  Trimming: Skipping the Transient Region

Plasma simulations go through a linear growth phase before reaching a
statistically stationary state (the *steady state* or *nonlinear saturation*).
Including the transient in a time average biases the result.

### The strategy-operation pattern

```
QuantileTrimStrategy   ──►  TrimDataStreamOperation  ──►  trimmed DataStream
  (decides WHERE)              (applies the cut)
```

The `start_time` keyword lets you enforce a *minimum* start time — the
algorithm will never begin the steady-state window earlier than this value,
even if the automatic detection would.

**Important:** always import from `quends.base.trim`, not from the top-level
package.

In [ ]:
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

# Skip the first 50 time units regardless of what the algorithm detects
strategy = QuantileTrimStrategy(window_size=20, start_time=50.0, robust=True)
op = TrimDataStreamOperation(strategy=strategy)
trimmed = op(ds_csv, column_name="HeatFlux_st")

print(f"Original  : t_start = {ds_csv.data['time'].iloc[0]:.2f}, "
      f"n_steps = {len(ds_csv.data)}")
print(f"Trimmed   : t_start = {trimmed.data['time'].iloc[0]:.2f}, "
      f"n_steps = {len(trimmed.data)}")

---

## 4  Building an Ensemble from Multiple CSV Files

When running parameter scans or seeded ensembles you will have multiple
simulation output files that share the same variable layout.  `quends.Ensemble`
aggregates them and exposes collective statistics.

The workflow:
1. Load each file into its own `DataStream`
2. Wrap them in `qnds.Ensemble`
3. Trim the whole ensemble in one call via `ens.trim()`

In [ ]:
files = [
    "gx/ensemble/tprim_2_5_a.out.csv",
    "gx/ensemble/tprim_2_5_b.out.csv",
    "gx/ensemble/tprim_2_5_c.out.csv",
]

members = [qnds.from_csv(f) for f in files]
ens = qnds.Ensemble(members)

# Trim all members at once using the "std" (QuantileTrim) method
trimmed_ens = ens.trim("HeatFlux_st", method="std", window_size=20)

print(f"Ensemble contains {len(trimmed_ens.data_streams)} trimmed members.")

---

## Summary

| Step | API |
|------|-----|
| Load CSV | `qnds.from_csv(path)` |
| Load NetCDF | `qnds.from_gx(path, variables=[...])` |
| Load dict / NumPy | `qnds.from_dict(d)` / `qnds.from_numpy(arr, columns=[...])` |
| Inspect columns | `ds.variables()` |
| Trim (single stream) | `TrimDataStreamOperation(QuantileTrimStrategy(...))(ds, column_name=...)` |
| Ensemble trim | `ens.trim(col, method="std", window_size=N)` |

**Next:** `08_postprocessing_workflow.ipynb` — statistics, ESS, confidence
intervals, and export.